In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Customer_Satisfaction').getOrCreate()
from sklearn.model_selection import train_test_split
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

In [2]:
data = pd.read_csv("preprocessed_data.csv")
data.drop("Unnamed: 0",axis=1,inplace=True)
data.head(5)

,payment_type,payment_installments,payment_value,review_score,customer_state,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,...,seller_state,distance,delivery_days,estimated_days,ships_in,review_time,arrival_time,delivery_impression,estimated_del_impression,ship_impression
0,3,8,99.33,1,0,921.0,8.0,800.0,17.0,27.0,...,0,845.34,14,27,7,5,1,2,0,2
1,3,1,24.39,5,1,1274.0,2.0,150.0,16.0,6.0,...,0,23.27,3,20,6,3,1,2,1,2
2,3,1,65.71,5,1,1536.0,2.0,250.0,20.0,8.0,...,0,27.31,6,23,14,3,1,2,1,1
3,3,8,107.78,5,0,188.0,1.0,1200.0,44.0,2.0,...,0,457.50,15,29,6,0,1,2,0,2
4,3,8,107.78,5,0,188.0,1.0,1200.0,44.0,2.0,...,0,457.50,15,29,6,1,1,2,0,2


In [3]:
X_train, X_test, y_train, y_test = train_test_split(data.drop('review_score', axis=1), data['review_score'], test_size=0.2, random_state=42)
train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)
train_data.to_csv("train_data.csv")
test_data.to_csv("test_data.csv")

In [4]:
train_data = spark.read.csv("train_data.csv", header=True, inferSchema=True)
test_data = spark.read.csv("test_data.csv", header=True, inferSchema=True)

In [5]:
cols = ['payment_type', 'payment_installments', 'payment_value',
       'customer_state', 'product_description_lenght', 'product_photos_qty',
       'product_weight_g', 'product_length_cm', 'product_height_cm',
       'product_width_cm', 'price', 'freight_value', 'seller_state',
       'distance', 'delivery_days', 'estimated_days', 'ships_in',
       'review_time', 'arrival_time', 'delivery_impression',
       'estimated_del_impression', 'ship_impression']

In [6]:
train_assembler = VectorAssembler(inputCols=cols, outputCol='features')
transformed_train_data = train_assembler.transform(train_data)
transformed_train_data.show(5, truncate=False)

test_assembler = VectorAssembler(inputCols=cols, outputCol='features')
transformed_test_data = test_assembler.transform(test_data) 
transformed_test_data.show(5, truncate=False)

+-----+------------+--------------------+-------------+--------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----+-------------+------------+--------+-------------+--------------+--------+-----------+------------+-------------------+------------------------+---------------+------------+-------------------------------------------------------------------------------------------------------------+
|_c0  |payment_type|payment_installments|payment_value|customer_state|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|price|freight_value|seller_state|distance|delivery_days|estimated_days|ships_in|review_time|arrival_time|delivery_impression|estimated_del_impression|ship_impression|review_score|features                                                                                                     |
+-----+------------+--------------------+-------

In [7]:
train_data = transformed_train_data.select('features', 'review_score')
train_data.show(5, truncate=False)

test_data = transformed_test_data.select('features', 'review_score') 
test_data.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------+------------+
|features                                                                                                     |review_score|
+-------------------------------------------------------------------------------------------------------------+------------+
|[2.0,1.0,61.71,1.0,1059.0,4.0,450.0,30.0,6.0,23.0,52.99,8.72,0.0,60.89,13.0,18.0,9.0,2.0,1.0,2.0,1.0,1.0]    |2           |
|[3.0,10.0,573.0,1.0,610.0,1.0,2550.0,30.0,35.0,30.0,75.0,21.2,0.0,118.06,15.0,22.0,6.0,1.0,1.0,2.0,1.0,2.0]  |1           |
|[2.0,1.0,58.13,8.0,1109.0,1.0,900.0,18.0,25.0,16.0,39.9,18.23,0.0,395.71,20.0,24.0,8.0,3.0,1.0,1.0,1.0,1.0]  |5           |
|[3.0,4.0,116.74,12.0,204.0,1.0,1663.0,36.0,14.0,27.0,98.8,17.94,0.0,500.49,11.0,22.0,4.0,1.0,1.0,2.0,1.0,2.0]|5           |
|[3.0,1.0,106.95,1.0,961.0,1.0,300.0,25.0,10.0,15.0,99.0,7.95,0.0,5.11,1.0,12.0,2.0,0.0,1.0,2.0,2.0,2.0]      |5           |


In [8]:
# Random Forrest

clf = RandomForestClassifier(labelCol='review_score', featuresCol='features')
clf = clf.fit(train_data)

predictions = clf.transform(test_data)
predictions.show(5, truncate=False)
predictions.select('review_score', 'prediction').show(5, truncate=False)

evaluator_multi = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='accuracy')
evaluator_weighted_precision = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedPrecision')
evaluator_weighted_recall = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedRecall')
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='f1')

accuracy = evaluator_multi.evaluate(predictions)
weighted_precision = evaluator_weighted_precision.evaluate(predictions)
weighted_recall = evaluator_weighted_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print(f'Accuracy: {accuracy}')
print(f'Weighted Precision: {weighted_precision}')
print(f'Weighted Recall: {weighted_recall}')
print(f'F1 Score: {f1}')

+---------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------+----------+
|features                                                                                                       |review_score|rawPrediction                                                                                      |probability                                                                                             |prediction|
+---------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------

In [9]:
# Logistic Regression

clf = LogisticRegression(labelCol='review_score', featuresCol='features')
clf = clf.fit(train_data)

predictions = clf.transform(test_data)
predictions.show(5, truncate=False)
predictions.select('review_score', 'prediction').show(5, truncate=False)

evaluator_multi = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='accuracy')
evaluator_weighted_precision = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedPrecision')
evaluator_weighted_recall = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedRecall')
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='f1')

accuracy = evaluator_multi.evaluate(predictions)
weighted_precision = evaluator_weighted_precision.evaluate(predictions)
weighted_recall = evaluator_weighted_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print(f'Accuracy: {accuracy}')
print(f'Weighted Precision: {weighted_precision}')
print(f'Weighted Recall: {weighted_recall}')
print(f'F1 Score: {f1}')

+---------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+----------+
|features                                                                                                       |review_score|rawPrediction                                                                                                        |probability                                                                                                                |prediction|
+---------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------------------------------------------------------------------------

In [10]:
# Decision Tree 

clf = DecisionTreeClassifier(labelCol='review_score', featuresCol='features')
clf = clf.fit(train_data)

predictions = clf.transform(test_data)
predictions.show(5, truncate=False)
predictions.select('review_score', 'prediction').show(5, truncate=False)

evaluator_multi = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='accuracy')
evaluator_weighted_precision = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedPrecision')
evaluator_weighted_recall = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedRecall')
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='f1')

accuracy = evaluator_multi.evaluate(predictions)
weighted_precision = evaluator_weighted_precision.evaluate(predictions)
weighted_recall = evaluator_weighted_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print(f'Accuracy: {accuracy}')
print(f'Weighted Precision: {weighted_precision}')
print(f'Weighted Recall: {weighted_recall}')
print(f'F1 Score: {f1}')

+---------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+--------------------------------------------------------------------------------------------------------+----------+
|features                                                                                                       |review_score|rawPrediction                             |probability                                                                                             |prediction|
+---------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+--------------------------------------------------------------------------------------------------------+----------+
|[3.0,3.0,214.27,6.0,1038.0,5.0,9075.0,29.0,39.0,45.0,195.0,19.27,1.0,970.37,12.0,37.0,8.0,2.0,1.0,2.0,0.0,1.0] |5           |[0.0,4580.0,1622